In [1]:
# ==========================
# 1. Bibliotecas e estrutura
# ==========================
import csv
import platform
import re
import sqlite3
from pathlib import Path

COLUNAS = [
    "ENTORNO", "PREFPE", "PRESPE", "PFEENT", "CODACT",
    "PCODCL", "PNOMCL", "PRUTA", "PCATCL", "CODTLI",
    "CODMOP", "PUNTOS", "PUNTOE", "PCAPIK", "CPOCLI",
    "DATM51", "JDEUNS", "JDECJS", "JDEPLS", "JDECPS",
    "JDERLS", "JDEKBS", "JDEKNS", "JDEUNE", "DATEXC",
    "UNIDAC",
]

COLUNA_DATA = "PFEENT"
COLUNA_ORIGEM = "ficheiro_origem"
COLUNAS_CHAVE = ("ENTORNO", "PREFPE", "PRESPE")
PREFIXO_TABELA = "ingresso_danone"
TAMANHO_BATCH = 10_000
REGEX_ANO = re.compile(r"^(\d{4})")


In [2]:
# ==========================
# 2. Caminhos
# ==========================
if platform.system() == "Windows":
    DB_PATH = Path(
        r"C:\Users\LISARR\Documents\python\01.Financeiro\inform_27.db"
    )
    PASTA_FICHEIROS = Path(
        r"C:\Users\LISARR\Documents\python\01.Financeiro\inform_27"
    )

elif platform.system() == "Darwin":
    DB_PATH = Path(
        "/Users/rr/Library/Mobile Documents/"
        "com~apple~CloudDocs/05.Salvesen/inform_27.db"
    )
    PASTA_FICHEIROS = Path(
        "/Users/rr/Library/Mobile Documents/"
        "com~apple~CloudDocs/05.Salvesen/kilospo"
    )

else:
    DB_PATH = Path("inform_27.db")
    PASTA_FICHEIROS = Path("inform_27")

DB_PATH.parent.mkdir(parents=True, exist_ok=True)

if not PASTA_FICHEIROS.exists():
    raise FileNotFoundError(
        f"Pasta não encontrada: {PASTA_FICHEIROS}"
    )

with sqlite3.connect(DB_PATH) as con:
    con.execute("PRAGMA journal_mode = WAL")

print(f"✓ BD: {DB_PATH}")
print(f"✓ Pasta: {PASTA_FICHEIROS}")


✓ BD: /Users/rr/Library/Mobile Documents/com~apple~CloudDocs/05.Salvesen/inform_27.db
✓ Pasta: /Users/rr/Library/Mobile Documents/com~apple~CloudDocs/05.Salvesen/kilospo


In [3]:
# ==========================
# 3. Funções auxiliares
# ==========================
def obter_ano(valor):
    if valor is None:
        return None

    texto = str(valor).strip()
    correspondencia = REGEX_ANO.match(texto)

    if not correspondencia or correspondencia.group(1) == "0000":
        return None

    return int(correspondencia.group(1))


def tabela_ano(ano):
    return f"{PREFIXO_TABELA}_{ano}"


def detectar_codificacao(caminho):
    for codificacao in ("utf-8-sig", "utf-8", "cp1252", "latin-1"):
        try:
            with caminho.open("r", encoding=codificacao, newline="") as ficheiro:
                ficheiro.read(100_000)
            return codificacao
        except UnicodeDecodeError:
            continue

    raise UnicodeError(
        f"Não foi possível identificar a codificação: {caminho.name}"
    )


def normalizar_cabecalho(nome):
    return str(nome).strip()


def normalizar_valor(valor):
    if valor is None:
        return None

    texto = str(valor).strip()
    return texto if texto else None


In [4]:
# ==========================
# 4. Estrutura SQLite
# ==========================
def criar_tabela(con, ano):
    tabela = tabela_ano(ano)
    colunas_sql = ", ".join(
        f'"{coluna}" TEXT'
        for coluna in COLUNAS
    )

    con.execute(
        f'''
        CREATE TABLE IF NOT EXISTS "{tabela}" (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            {colunas_sql},
            "{COLUNA_ORIGEM}" TEXT NOT NULL
        )
        '''
    )

    con.execute(
        f'''
        CREATE UNIQUE INDEX IF NOT EXISTS "idx_{tabela}_chave"
        ON "{tabela}" ("ENTORNO", "PREFPE", "PRESPE")
        '''
    )

    con.execute(
        f'''
        CREATE INDEX IF NOT EXISTS "idx_{tabela}_pfeent"
        ON "{tabela}" ("PFEENT")
        '''
    )

    con.execute(
        f'''
        CREATE INDEX IF NOT EXISTS "idx_{tabela}_pcodcl"
        ON "{tabela}" ("PCODCL")
        '''
    )

    return tabela


def inserir_linhas(con, ano, linhas):
    if not linhas:
        return 0, 0

    tabela = criar_tabela(con, ano)
    nomes_colunas = [*COLUNAS, COLUNA_ORIGEM]
    colunas_sql = ", ".join(
        f'"{coluna}"'
        for coluna in nomes_colunas
    )
    placeholders = ", ".join("?" for _ in nomes_colunas)

    sql = (
        f'INSERT OR IGNORE INTO "{tabela}" '
        f'({colunas_sql}) VALUES ({placeholders})'
    )

    antes = con.total_changes
    con.executemany(sql, linhas)
    inseridos = con.total_changes - antes

    return inseridos, len(linhas) - inseridos


In [5]:
# ==========================
# 5. Importação dos CSV
# ==========================
ficheiros = sorted(
    caminho
    for caminho in PASTA_FICHEIROS.rglob("*")
    if caminho.is_file() and caminho.suffix.lower() == ".csv"
)

if not ficheiros:
    raise FileNotFoundError(
        f"Nenhum ficheiro CSV em: {PASTA_FICHEIROS}"
    )

totais = {
    "ficheiros": 0,
    "linhas": 0,
    "inseridos": {},
    "duplicados": 0,
    "sem_data": 0,
    "sem_chave": 0,
    "erros": 0,
}

con = sqlite3.connect(DB_PATH)
con.execute("PRAGMA journal_mode = WAL")
con.execute("PRAGMA synchronous = NORMAL")
con.execute("PRAGMA busy_timeout = 30000")

for numero, caminho in enumerate(ficheiros, start=1):
    try:
        codificacao = detectar_codificacao(caminho)
        estatisticas = {
            "linhas": 0,
            "novos": 0,
            "duplicados": 0,
            "sem_data": 0,
            "sem_chave": 0,
        }
        batches = {}

        with caminho.open(
            "r",
            encoding=codificacao,
            newline="",
        ) as ficheiro:
            reader = csv.DictReader(ficheiro, delimiter=";")

            if reader.fieldnames is None:
                raise ValueError("CSV sem cabeçalho.")

            reader.fieldnames = [
                normalizar_cabecalho(coluna)
                for coluna in reader.fieldnames
            ]

            colunas_em_falta = [
                coluna
                for coluna in (*COLUNAS_CHAVE, COLUNA_DATA)
                if coluna not in reader.fieldnames
            ]

            if colunas_em_falta:
                raise ValueError(
                    f"Colunas obrigatórias em falta: {colunas_em_falta}"
                )

            for registo in reader:
                estatisticas["linhas"] += 1
                registo = {
                    normalizar_cabecalho(coluna): normalizar_valor(valor)
                    for coluna, valor in registo.items()
                    if coluna is not None
                }

                ano = obter_ano(registo.get(COLUNA_DATA))

                if ano is None:
                    estatisticas["sem_data"] += 1
                    continue

                if any(not registo.get(coluna) for coluna in COLUNAS_CHAVE):
                    estatisticas["sem_chave"] += 1
                    continue

                linha = [registo.get(coluna) for coluna in COLUNAS]
                linha.append(caminho.name)
                batches.setdefault(ano, []).append(linha)

                if len(batches[ano]) >= TAMANHO_BATCH:
                    inseridos, duplicados = inserir_linhas(
                        con,
                        ano,
                        batches[ano],
                    )
                    estatisticas["novos"] += inseridos
                    estatisticas["duplicados"] += duplicados
                    totais["inseridos"][ano] = (
                        totais["inseridos"].get(ano, 0) + inseridos
                    )
                    batches[ano].clear()

        for ano, batch in sorted(batches.items()):
            inseridos, duplicados = inserir_linhas(
                con,
                ano,
                batch,
            )
            estatisticas["novos"] += inseridos
            estatisticas["duplicados"] += duplicados
            totais["inseridos"][ano] = (
                totais["inseridos"].get(ano, 0) + inseridos
            )

        con.commit()
        totais["ficheiros"] += 1
        totais["linhas"] += estatisticas["linhas"]
        totais["duplicados"] += estatisticas["duplicados"]
        totais["sem_data"] += estatisticas["sem_data"]
        totais["sem_chave"] += estatisticas["sem_chave"]

        print(
            f"[{numero}/{len(ficheiros)}] {caminho.name} | "
            f"+{estatisticas['novos']:,} novos | "
            f"{estatisticas['duplicados']:,} duplicados | "
            f"{estatisticas['sem_data']:,} sem data | "
            f"{estatisticas['sem_chave']:,} sem chave"
        )

    except Exception as erro:
        con.rollback()
        totais["erros"] += 1
        print(
            f"[{numero}/{len(ficheiros)}] "
            f"{caminho.name} — ERRO: {erro}"
        )

con.close()

print("\n--- RESUMO ---")
print(
    f"Ficheiros: {totais['ficheiros']} | "
    f"Linhas lidas: {totais['linhas']:,} | "
    f"Duplicados: {totais['duplicados']:,} | "
    f"Sem data: {totais['sem_data']:,} | "
    f"Sem chave: {totais['sem_chave']:,} | "
    f"Erros: {totais['erros']}"
)

for ano, quantidade in sorted(totais["inseridos"].items()):
    print(
        f"Linhas novas em {tabela_ano(ano)}: "
        f"{quantidade:,}"
    )


[1/4] INFKILP (2).CSV | +0 novos | 21,814 duplicados | 1 sem data | 0 sem chave
[2/4] INFKILP (3).CSV | +0 novos | 44,604 duplicados | 1 sem data | 0 sem chave
[3/4] INFKILP_2026_PTG.CSV | +42,436 novos | 498 duplicados | 1 sem data | 0 sem chave
[4/4] INFKILP_2026_aza.CSV | +13,636 novos | 164 duplicados | 1 sem data | 0 sem chave

--- RESUMO ---
Ficheiros: 4 | Linhas lidas: 123,156 | Duplicados: 67,080 | Sem data: 4 | Sem chave: 0 | Erros: 0
Linhas novas em ingresso_danone_2023: 0
Linhas novas em ingresso_danone_2024: 0
Linhas novas em ingresso_danone_2025: 0
Linhas novas em ingresso_danone_2026: 56,072


In [6]:
# ==========================
# 6. Validação final
# ==========================
with sqlite3.connect(DB_PATH) as con:
    tabelas = con.execute(
        """
        SELECT name
        FROM sqlite_master
        WHERE type = 'table'
          AND name LIKE 'ingresso_danone_%'
        ORDER BY name
        """
    ).fetchall()

    resultado = []

    for (tabela,) in tabelas:
        total = con.execute(
            f'SELECT COUNT(*) FROM "{tabela}"'
        ).fetchone()[0]

        resultado.append(
            {
                "tabela": tabela,
                "registos": total,
            }
        )

resultado


[{'tabela': 'ingresso_danone_2023', 'registos': 381},
 {'tabela': 'ingresso_danone_2024', 'registos': 44307},
 {'tabela': 'ingresso_danone_2025', 'registos': 22218},
 {'tabela': 'ingresso_danone_2026', 'registos': 56072}]